# Scraping Booking for Primavera Sound

In [135]:
# Core imports
import pandas as pd
import numpy as np
import requests
import time
import random
import re
from datetime import datetime, timedelta

# Selenium imports
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    TimeoutException, 
    NoSuchElementException,
    StaleElementReferenceException
)

# For automatic chromedriver management
from webdriver_manager.chrome import ChromeDriverManager

# For parsing HTML
from bs4 import BeautifulSoup

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# import datetime
from datetime import datetime

import pandas as pd

## Open the Booking.com in workable condition

In [2]:
def open_website(website_url, headless_bool=False):
    # Create a simple driver (WITH visible browser window)
    options = Options()

    if headless_bool:
        options.add_argument("--headless=new")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--incognito")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    print("✅ Browser opened!")
    time.sleep(3)  # Wait for page to load

    driver.get(website_url)
    time.sleep(3)  # Wait for page to load

    print("✅ Navigated to Booking.com search results")
    print(f"Page title: {driver.title}")

    driver.set_window_size(1400, 900)

    # accept cookies
    try:
        accept_button = driver.find_element(By.ID, "onetrust-accept-btn-handler")
        accept_button.click()  # Example: Accept cookies button
        time.sleep(5)  # Wait for page to load
    except:
        pass

    try:
        wait = WebDriverWait(driver, 10)
        close_signin = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "button[aria-label='Dismiss sign-in info.']")
            )
        )
        close_signin.click()
    except:
        pass
    
    return driver


## Search the chosen city and dates

In [13]:
def search(driver, city, start_date, end_date):

    wait = WebDriverWait(driver, 10)

    # type in selected city
    city_input = driver.find_element(By.NAME, "ss")
    city_input.clear()
    city_input.send_keys(city)
    time.sleep(2)

    js = """
    document.addEventListener("mousemove", e => {
        const box = document.getElementById("__coords__") || (() => {
            const d = document.createElement("div");
            d.id = "__coords__";
            d.style.cssText = `
                position:fixed;top:0;left:0;z-index:999999;
                background:black;color:lime;
                font:12px monospace;padding:4px;
                pointer-events:none;
            `;
            document.body.appendChild(d);
            return d;
        })();

        box.textContent = `x:${e.clientX} y:${e.clientY}`;
    });
    """

    driver.execute_script(js)

    # select the city
    x = 250
    y = 420
    try:
        # Try clicking the first autocomplete result
        driver.find_element(By.CSS_SELECTOR, "ul[data-testid='autocomplete-results'] li:first-child").click()
    except:
        driver.execute_script(f"document.elementFromPoint({x}, {y}).click();")
    
    time.sleep(2)

    # scroll to necessary month
    try:
        next_btn = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Next month']"))
        )
    except:
         # Open date picker if not open
         driver.find_element(By.CSS_SELECTOR, "div[data-testid='searchbox-dates-container']").click()
         next_btn = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Next month']"))
        )

    start = datetime.strptime(start_date, "%Y-%m-%d")
    today = datetime.today()
    # Ensure we don't look at past. But start_date is 2026. Today is 2026-01-19.
    months = (start.year - today.year) * 12 + (start.month - today.month)

    for _ in range(months):
        time.sleep(0.5)
        next_btn.click()

    # select dates
    wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, f"span[data-date='{start_date}']"))
    ).click()

    time.sleep(1)
    
    wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, f"span[data-date='{end_date}']"))
    ).click()

    # click search
    for span in driver.find_elements(By.TAG_NAME, "span"):
        if span.text.strip().lower() == "search":
            span.click()
            break
    
    print("✅ Search Complete")    
    return driver

## Get the full list of listing

In [ ]:
def collect_all_hotels(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(2)
    
    hotel_data = {}  # Use dict to avoid duplicates by link
    last_count = 0
    stagnation = 0
    MAX_STAGNATION = 10
    SCROLL_STEP = 900
    SCROLL_PAUSE = 4
    
    print("Starting scrolling ...")
    
    while True:
        # Scroll
        driver.execute_script(f"window.scrollBy(0, {SCROLL_STEP});")
        time.sleep(SCROLL_PAUSE)
        
        # Click "Load more"
        try:
            btn = driver.find_element(By.XPATH, "//span[text()='Load more results']")
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(2)
            btn.click()
            time.sleep(5)
        except:
            pass
        
        # Collect hotels
        cards = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='property-card']")
        for card in cards:
            try:
                # Extract link
                link = card.find_element(By.CSS_SELECTOR, "a[data-testid='property-card-desktop-single-image']").get_attribute("href")
                
                if not link or link in hotel_data:
                    continue
                
                # Extract title
                try:
                    title = card.find_element(By.CSS_SELECTOR, "div[data-testid='title']").text
                except:
                    title = None
                
                # Extract price
                try:
                    price = card.find_element(
                        By.CSS_SELECTOR, "span[data-testid='price-and-discounted-price']"
                    ).text
                except:
                    price = None
                
                # Store data
                hotel_data[link] = {'link': link, 'hotel': title, 'price': price}
                
            except:
                continue
        
        current = len(hotel_data)
        print(f"Unique hotels: {current}")
        
        # Stop condition (data-based)
        if current == last_count:
            stagnation += 1
            # print(f"No growth ({stagnation}/{MAX_STAGNATION})")
        else:
            stagnation = 0
        
        last_count = current
        
        if stagnation >= MAX_STAGNATION:
            print("No more hotels loading — confirmed.")
            break
    
    print(f"FINAL HOTEL COUNT: {len(hotel_data)}")
    
    # Convert to DataFrame
    df = pd.DataFrame.from_dict(hotel_data, orient='index')
    df = df.reset_index(drop=True)
    
    return driver, df

## extract description

In [127]:
def extract_description(driver, link):
    # Navigate to the link
    driver.get(link)

    # Maximize the window to make sure it's visible
    driver.maximize_window()

    # Wait for page to load
    time.sleep(3)

    # Bring window to front (different methods for different OS)
    try:
        driver.switch_to.window(driver.current_window_handle)
    except:
        pass

    # Verify we're on the right page
    # print(f"Navigating to: {driver.title}")

    # Extract the description
    try:
        description = driver.find_element(By.CSS_SELECTOR, "p[data-testid='property-description']").text
        # print("Description extracted\n")    
    except Exception as e:
        # print(f"Error extracting description: {e}")
        description = None
    return description

## Run the process flow

In [ ]:
# --- Configuration ---
city = "Alicante"
booking_url = "https://www.booking.com/?lang=en-us&selected_currency=EUR"
start_date_range = datetime.strptime("2026-02-27", "%Y-%m-%d")
end_date_range = datetime.strptime("2026-03-28", "%Y-%m-%d")

# Initialize master storage
master_results = []
# Cache for descriptions to avoid re-opening the same hotel page on different dates
descriptions_cache = {}

print(f"Starting Scraping for {city} from {start_date_range.date()} to {end_date_range.date()}")

# Open browser once
driver = open_website(website_url=booking_url)

current_start = start_date_range
while current_start <= end_date_range:
    checkin_s = current_start.strftime("%Y-%m-%d")
    checkout_s = (current_start + timedelta(days=7)).strftime("%Y-%m-%d")
    
    print(f"\n SCRAPING PERIOD: {checkin_s} to {checkout_s} (Stay: 7 nights)")
    
    try:
        # Navigate to home to reset the search interface for the search function
        driver.get(booking_url)
        time.sleep(3)
        
        # 1. Perform Search for the specific date pair
        driver = search(driver=driver, city=city, start_date=checkin_s, end_date=checkout_s)
        
        # --- APPLY FILTER FOR HOTELS (ht_id=204) IF NOT PRESENT ---
        curr_url = driver.current_url
        if "ht_id=204" not in curr_url:
            print("Applying Hotel filter...")
            if "?" in curr_url:
                new_url = curr_url + "&nflt=ht_id%3D204"
            else:
                new_url = curr_url + "?nflt=ht_id%3D204"
            driver.get(new_url)
            time.sleep(4)
        
        # 2. Collect all hotel listings found for this period
        driver, daily_df = collect_all_hotels(driver)
        
        if not daily_df.empty:
            # 3. Add metadata to the daily results
            daily_df['checkin_date'] = checkin_s
            daily_df['city'] = city
            # Clean price
            if 'price' in daily_df.columns:
                 daily_df['price'] = daily_df['price'].str.replace("€ ", "").str.replace(",", "").astype(float, errors='ignore')
            
            # 4. Extract sample descriptions (n=3 per original setup)
            n_sample = 3
            daily_df['text'] = None
            for i in range(min(n_sample, len(daily_df))):
                link = daily_df.loc[i, 'link']
                # Use cache if we've already seen this hotel to save time
                if link not in descriptions_cache:
                    # Sleep a bit before request
                    time.sleep(random.randint(2, 5))
                    descriptions_cache[link] = extract_description(driver, link)
                
                daily_df.at[i, 'text'] = descriptions_cache[link]
            
            master_results.append(daily_df)
            
            # 5. Persistent Save: Update CSV after every day
            save_path = 'booking_hotels_alicante_full_scrape.csv'
            pd.concat(master_results).to_csv(save_path, index=False)
            print(f"✅ Successfully saved results for {checkin_s}. Total rows: {len(pd.concat(master_results))}")
        
    except Exception as e:
        print(f"❌ Error processing date {checkin_s}: {e}")
        # Optional: Save current progress even on error
        if master_results:
            pd.concat(master_results).to_csv('booking_hotels_alicante_interrupted.csv', index=False)

    # Move to the next day
    current_start += timedelta(days=1)

driver.quit()
print(f"\n🎉 ALL SEARCHES COMPLETED. Final data saved to {save_path}")

Starting Scraping for Alicante from 2026-02-27 to 2026-03-28
✅ Browser opened!
✅ Navigated to Booking.com search results
Page title: Booking.com | Official site | The best hotels, flights, car rentals & accommodations

🚀 SCRAPING PERIOD: 2026-02-27 to 2026-03-06 (Stay: 7 nights)
✅ Search Complete
Applying Hotel filter...
Starting scrolling ...
Unique hotels: 25
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
Unique hotels: 50
No more hotels loading — confirmed.
FINAL HOTEL COUNT: 50
✅ Successfully saved results for 2026-02-27. Total rows: 50

🚀 SCRAPING PERIOD: 2026-02-28 to 2026-03-07 (Stay: 7 nights)
❌ Error processing date 2026-02-28: Message: no such element: Unable to locate element: {"method":"css selector","selector":"div[data-testid='searchbox-dates-container']"}
  (Session info: chrome=143.0.7499.193); For documentation on this error, please visit